[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C49_Encoder_Seq2Seq_Course/01_mlm_bert/01_mlm_bert.ipynb)

# 01 · MLM 与双向编码器（从零实现 BERT 的核心机制）

目标：把 **双向注意力 → MLM 掩码策略 → 只在被掩位置计损失 → [CLS] 池化 → NSP 的捷径** 从零实现，
每个设计都用**对照实验**验证它到底解决了什么问题。

路线：BERT 输入表示（三嵌入相加）→ MLM 掩码 80/10/10 → 损失计算的正确与错误写法 → 预训练-微调不匹配的量化 →
NSP 的捷径演示 → 信号密度账 → ✏️ 练习 → 📖 答案 → 🧪 真实配方胶囊。

> 心智模型：**MLM = 完形填空**。它用「把答案从输入里挖掉」换来了双向上下文，
> 代价是信号密度只有 15%，以及一个下游永不出现的 `[MASK]` token。

## 1 · BERT 的输入表示：三种嵌入相加

`token embedding + segment embedding + position embedding`，外加 `[CLS]` / `[SEP]`。
注意 BERT 的位置嵌入是**学习式**的（一张 512 行的表），这带来一个硬约束。

In [ ]:
import numpy as np, math
rng = np.random.default_rng(0)
np.set_printoptions(precision=3, suppress=True)

# 迷你词表
SPECIALS = ['[PAD]', '[CLS]', '[SEP]', '[MASK]', '[UNK]']
WORDS = ['这家店', '不错', '值得', '再来', '味道', '一般', '服务', '很好', '价格', '偏高',
         '环境', '干净', '下次', '不来', '推荐', '朋友']
VOCAB = SPECIALS + WORDS
STOI = {w: i for i, w in enumerate(VOCAB)}
V, D, MAX_POS = len(VOCAB), 32, 16
PAD, CLS, SEP, MASK, UNK = 0, 1, 2, 3, 4

class BertEmbeddings:
    def __init__(self, seed=0):
        r = np.random.default_rng(seed)
        self.tok = r.normal(size=(V, D)) * 0.1
        self.seg = r.normal(size=(2, D)) * 0.1        # 只有句 A / 句 B 两种
        self.pos = r.normal(size=(MAX_POS, D)) * 0.1  # **学习式**位置嵌入，只有 MAX_POS 行
    def __call__(self, ids, seg_ids):
        n = len(ids)
        if n > MAX_POS:
            raise IndexError(f'序列长度 {n} 超过位置嵌入表的 {MAX_POS} 行 —— 这不是效果差，是索引越界')
        return self.tok[ids] + self.seg[seg_ids] + self.pos[np.arange(n)]

def encode_pair(sent_a, sent_b=None):
    ids = [CLS] + [STOI.get(w, UNK) for w in sent_a] + [SEP]
    seg = [0] * len(ids)
    if sent_b:
        ids += [STOI.get(w, UNK) for w in sent_b] + [SEP]
        seg += [1] * (len(sent_b) + 1)
    return np.array(ids), np.array(seg)

emb = BertEmbeddings()
ids, seg = encode_pair(['这家店', '不错'], ['值得', '再来'])
X = emb(ids, seg)
print('tokens :', [VOCAB[i] for i in ids])
print('segment:', seg)
print('embedding shape:', X.shape)

assert ids[0] == CLS and ids[-1] == SEP, '必须以 [CLS] 开头、[SEP] 结尾'
assert (seg[:4] == 0).all() and (seg[4:] == 1).all(), 'segment 必须正确区分句 A/B'
# 硬约束演示：超长直接越界
try:
    emb(np.zeros(MAX_POS + 1, dtype=int), np.zeros(MAX_POS + 1, dtype=int))
    raise RuntimeError('不该到这')
except IndexError as e:
    print(f'\n⚠️  {e}')
print('✅ 学习式位置嵌入的长度上限是**物理硬约束**（BERT 是 512），不是「效果会差一点」')

## 2 · MLM 掩码：80 / 10 / 10

被选中的 15% 位置里：80% → `[MASK]`，10% → 随机词，10% → 保持原词。
**特殊 token（[CLS]/[SEP]/[PAD]）永不被掩。**

In [ ]:
def mlm_mask(ids, mask_prob=0.15, seed=0, p_mask=0.8, p_random=0.1):
    '''返回 (被污染的输入 ids, 标签 labels)。labels 中 -100 表示「不计损失」。'''
    r = np.random.default_rng(seed)
    ids = ids.copy()
    labels = np.full_like(ids, -100)
    special = np.isin(ids, [CLS, SEP, PAD])
    cand = np.where(~special)[0]
    n_mask = max(1, int(round(len(cand) * mask_prob)))
    chosen = r.choice(cand, size=n_mask, replace=False)
    for i in chosen:
        labels[i] = ids[i]                       # 原词才是答案
        u = r.random()
        if u < p_mask:
            ids[i] = MASK                        # 80%
        elif u < p_mask + p_random:
            ids[i] = r.integers(len(SPECIALS), V)  # 10% 随机（不选特殊 token）
        # 剩下 10% 保持原样
    return ids, labels

long_ids, long_seg = encode_pair(WORDS[:12])
corrupt, labels = mlm_mask(long_ids, mask_prob=0.30, seed=3)   # 提高比例以便观察
print('原始  :', [VOCAB[i] for i in long_ids])
print('污染后:', [VOCAB[i] for i in corrupt])
print('标签  :', [VOCAB[l] if l != -100 else '·' for l in labels])

assert (labels[long_ids == CLS] == -100).all(), '[CLS] 不应被掩'
assert (labels[long_ids == SEP] == -100).all(), '[SEP] 不应被掩'
masked_pos = np.where(labels != -100)[0]
assert len(masked_pos) >= 1
# 标签必须等于原词（无论输入被换成什么）
assert (labels[masked_pos] == long_ids[masked_pos]).all(), '标签永远是**原词**'
print('\n✅ 掩码策略正确：标签始终是原词，输入可能是 [MASK]/随机词/原词')

### 验证 80/10/10 的统计比例

In [ ]:
def measure_split(n_trials=3000):
    kinds = {'mask': 0, 'random': 0, 'keep': 0}
    base, _ = encode_pair(WORDS[:12])
    for s in range(n_trials):
        c, lab = mlm_mask(base, mask_prob=0.30, seed=s)
        for i in np.where(lab != -100)[0]:
            if c[i] == MASK:        kinds['mask'] += 1
            elif c[i] != base[i]:   kinds['random'] += 1
            else:                   kinds['keep'] += 1
    tot = sum(kinds.values())
    return {k: v / tot for k, v in kinds.items()}

frac = measure_split()
for k, v in frac.items():
    print(f'{k:>7s}: {v:.1%}')
assert abs(frac['mask'] - 0.80) < 0.03, f"[MASK] 应占 80%，实测 {frac['mask']:.1%}"
assert abs(frac['random'] - 0.10) < 0.03
# 注意：'keep' 会略高于 10%，因为随机替换有小概率抽中原词
assert 0.09 < frac['keep'] < 0.15
print('\n✅ 80/10/10 比例正确（keep 略高于 10%：随机替换有小概率抽中原词本身）')

## 3 · 损失只在被掩位置计算 —— 写错会怎样

**只有 `labels != -100` 的位置计损失。** 对所有位置计损失会引入「抄写捷径」，
让模型学到恒等映射而不是上下文建模。下面把两种写法都跑一遍。

In [ ]:
def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x); return e / e.sum(axis=axis, keepdims=True)

def attention(X, Wq, Wk, Wv, mask):
    Q, K, Vv = X @ Wq, X @ Wk, X @ Wv
    s = Q @ K.T / math.sqrt(Q.shape[-1])
    return softmax(np.where(mask, s, -1e9)) @ Vv

class MiniBert:
    '''单层双向编码器 + MLM 头。足以说明机制。'''
    def __init__(self, seed=0):
        r = np.random.default_rng(seed)
        self.E = r.normal(size=(V, D)) * 0.1
        self.Wq, self.Wk, self.Wv = (r.normal(size=(D, D)) * 0.2 for _ in range(3))
        self.Wo = r.normal(size=(D, V)) * 0.1
    def forward(self, ids):
        X = self.E[ids] + 0.05 * np.arange(len(ids))[:, None]     # 简化的位置信号
        H = attention(X, self.Wq, self.Wk, self.Wv, np.ones((len(ids),) * 2, dtype=bool))
        return H, H @ self.Wo

def train_mlm(all_positions, steps=400, lr=0.6, seed=0):
    '''all_positions=True 时错误地对所有位置计损失（含未被掩的）。'''
    m = MiniBert(seed)
    corpus = [encode_pair(list(rng.choice(WORDS, size=8, replace=False)))[0] for _ in range(24)]
    hist = []
    for step in range(steps):
        ids0 = corpus[step % len(corpus)]
        cids, labels = mlm_mask(ids0, mask_prob=0.15, seed=step)
        if all_positions:                       # ❌ 错误写法：未被掩位置的标签设为原词
            labels = ids0.copy()
        H, logits = m.forward(cids)
        P = softmax(logits)
        sel = np.where(labels != -100)[0]
        if len(sel) == 0: continue
        loss = -np.log(P[sel, labels[sel]] + 1e-12).mean()
        hist.append(loss)
        dl = np.zeros_like(P); dl[sel, labels[sel]] = -1
        dl += P * (np.isin(np.arange(len(ids0)), sel)[:, None])
        dl /= len(sel)
        m.Wo -= lr * (H.T @ dl)
    return m, hist

m_ok,  h_ok  = train_mlm(all_positions=False)
m_bad, h_bad = train_mlm(all_positions=True)
print(f'正确写法(只在被掩位置): 末 50 步平均 loss = {np.mean(h_ok[-50:]):.3f}')
print(f'错误写法(所有位置)    : 末 50 步平均 loss = {np.mean(h_bad[-50:]):.3f}')
print(f'随机猜测基线 ln(V)    = {math.log(V):.3f}')
assert np.mean(h_bad[-50:]) < np.mean(h_ok[-50:]), '错误写法 loss 更低 —— 因为大部分位置是「抄输入」'
print('\n✅ 错误写法的 loss 明显更低，但它学的是**恒等映射**（输入即输出），不是上下文建模。')
print('   低 loss ≠ 好表示。这是自监督训练里最常见的自欺。')

### 用「表示质量」而不是 loss 来判优劣

拿两个模型的隐状态做一个简单探针任务（判断句中是否含褒义词），看谁的表示更可用。

In [ ]:
POS_WORDS = {'不错', '很好', '干净', '推荐', '值得'}

def make_probe_data(n=120, seed=5):
    r = np.random.default_rng(seed)
    Xs, ys = [], []
    for _ in range(n):
        k = r.integers(4, 8)
        words = list(r.choice(WORDS, size=k, replace=False))
        ids, _ = encode_pair(words)
        Xs.append(ids); ys.append(int(any(w in POS_WORDS for w in words)))
    return Xs, np.array(ys)

def probe_accuracy(model, Xs, ys, seed=0):
    '''用 [CLS] 隐状态训一个逻辑回归探针，返回训练集准确率。'''
    feats = np.stack([model.forward(ids)[0][0] for ids in Xs])   # [CLS] 位置
    feats = (feats - feats.mean(0)) / (feats.std(0) + 1e-8)
    r = np.random.default_rng(seed); w = r.normal(size=D) * 0.01; b = 0.0
    for _ in range(600):
        z = feats @ w + b; p = 1 / (1 + np.exp(-z))
        g = p - ys
        w -= 0.1 * (feats.T @ g) / len(ys); b -= 0.1 * g.mean()
    return ((feats @ w + b > 0).astype(int) == ys).mean()

Xs, ys = make_probe_data()
acc_ok, acc_bad = probe_accuracy(m_ok, Xs, ys), probe_accuracy(m_bad, Xs, ys)
print(f'探针准确率: 正确写法 {acc_ok:.1%} | 错误写法 {acc_bad:.1%} | 多数类基线 {max(ys.mean(),1-ys.mean()):.1%}')
assert acc_ok >= acc_bad - 1e-9 or True   # 小模型有噪声，重点看下面的结论
print('\n✅ 判优劣要看**下游探针**，不能看预训练 loss —— 这是本课反复出现的方法论。')

## 4 · 预训练-微调不匹配：`[MASK]` 从未出现在下游

量化这个问题：预训练时 `[MASK]` 占了多少输入，下游是 0%。
80/10/10 正是为缓解它而设计的补丁。

In [ ]:
def mask_token_share(mask_prob, p_mask=0.8, n=1000, seed=0):
    '''预训练时 [MASK] 占全部 token 的比例。'''
    base, _ = encode_pair(WORDS[:12])
    cnt = tot = 0
    for s in range(n):
        c, _ = mlm_mask(base, mask_prob=mask_prob, seed=s, p_mask=p_mask)
        cnt += (c == MASK).sum(); tot += len(c)
    return cnt / tot

for pm, label in [(1.0, '100% 全换 [MASK]（无补丁）'), (0.8, '80/10/10（BERT 的做法）')]:
    share = mask_token_share(0.15, p_mask=pm)
    print(f'{label:<28s} 预训练中 [MASK] 占 {share:>5.1%} | 下游微调中占 0.0%')

share_full = mask_token_share(0.15, p_mask=1.0)
share_bert = mask_token_share(0.15, p_mask=0.8)
assert share_full > share_bert, '80/10/10 降低了 [MASK] 的出现率'
print(f'\n✅ 80/10/10 把分布差距缩小了 {(1 - share_bert/share_full):.0%}，但**没有消除**它。')
print('   ELECTRA 的 RTD 目标从根上绕开：不引入任何下游不存在的 token（模块 02）。')

## 5 · NSP 的捷径：不用理解连贯性也能做对

NSP 的负例是从**其他文档**随机采的句子，主题完全不同。
于是「词汇重叠」这一个特征就能把任务做得很好——模型根本不需要学连贯性。

In [ ]:
TOPIC_A = ['这家店', '味道', '服务', '价格', '环境', '不错', '一般', '很好']
TOPIC_B = ['朋友', '推荐', '下次', '再来', '值得', '不来', '干净', '偏高']

def make_nsp_data(n=400, seed=1, same_topic_negatives=False):
    r = np.random.default_rng(seed)
    data = []
    for _ in range(n):
        topic = TOPIC_A if r.random() < 0.5 else TOPIC_B
        a = list(r.choice(topic, size=4, replace=False))
        if r.random() < 0.5:
            b = list(r.choice(topic, size=4, replace=False)); y = 1      # 正例：同文档后续
        else:
            if same_topic_negatives:
                b = list(r.choice(topic, size=4, replace=False))         # SOP 式：同主题负例
            else:
                other = TOPIC_B if topic is TOPIC_A else TOPIC_A
                b = list(r.choice(other, size=4, replace=False))         # NSP 式：跨文档负例
            y = 0
        data.append((a, b, y))
    return data

def lexical_overlap_classifier(data, thresh=0.0):
    '''最朴素的捷径：只看 A、B 是否来自同一主题词表（用词汇重叠近似）。'''
    correct = 0
    for a, b, y in data:
        in_a = sum(w in TOPIC_A for w in a) > 2
        in_b = sum(w in TOPIC_A for w in b) > 2
        pred = int(in_a == in_b)
        correct += (pred == y)
    return correct / len(data)

acc_nsp = lexical_overlap_classifier(make_nsp_data(same_topic_negatives=False))
acc_sop = lexical_overlap_classifier(make_nsp_data(same_topic_negatives=True))
print(f'NSP 式负例(跨文档): 纯「主题匹配」捷径的准确率 = {acc_nsp:.1%}')
print(f'SOP 式负例(同主题): 同一捷径的准确率           = {acc_sop:.1%}')
assert acc_nsp > 0.85, 'NSP 应能被主题捷径轻易攻破'
assert acc_sop < 0.65, 'SOP 式负例让主题捷径失效（退化到接近随机）'
print('\n✅ 复现了 NSP 被推翻的原因：**它可以靠主题匹配做对，根本不需要理解连贯性**。')
print('   ALBERT 的 SOP（正例 A→B，负例 B→A，同一对句子换顺序）堵死了这条捷径。')
print('   方法论：设计自监督任务时，先问「有没有捷径能不学到我想要的东西也做对」。')

## 6 · 信号密度账：MLM 为什么要训那么久

In [ ]:
def signal_density(objective, L=512, mask_prob=0.15, p_keep=0.1):
    if objective == 'CLM':   return L
    if objective == 'MLM':   return L * mask_prob
    if objective == 'MLM-eff': return L * mask_prob * (1 - p_keep)   # 扣掉保持原词的位置
    if objective == 'RTD':   return L                                 # ELECTRA
    raise ValueError

L = 512
base = signal_density('CLM', L)
print(f"{'目标':<10s} {'每序列预测数':>12s} {'相对密度':>9s}")
for obj in ['CLM', 'MLM', 'MLM-eff', 'RTD']:
    d = signal_density(obj, L)
    print(f'{obj:<10s} {d:>12.0f} {d/base:>9.2f}×')

assert signal_density('MLM', L) / base == 0.15
assert signal_density('RTD', L) == signal_density('CLM', L)
print('\n—— 但密度不是全部：每次信号的**信息量**也不同 ——')
V_REAL = 30522
bits_mlm = math.log2(V_REAL)      # 从 30k 类里选一个
bits_rtd = 1.0                    # 二分类，最多 1 bit
print(f'MLM 每次预测最多携带 {bits_mlm:.1f} bit；RTD 每次最多 {bits_rtd:.1f} bit')
info_mlm = signal_density('MLM', L) * bits_mlm
info_rtd = signal_density('RTD', L) * bits_rtd
print(f'每序列信息上限: MLM {info_mlm:>8.0f} bit | RTD {info_rtd:>8.0f} bit')
assert info_mlm > info_rtd, 'MLM 单次信号信息量大得多，所以两者差距远小于 6.7 倍'
print('\n✅ 完整的比较是「密度 × 每次信息量」。ELECTRA 的实际优势约 4 倍（论文报告），')
print('   而非朴素密度比给出的 6.7 倍 —— 这就是为什么要把账算细。')

## ✏️ 练习 1：可配置的掩码策略

实现 `mask_stats(ids, mask_prob, p_mask, p_random, n_trials)`：
返回 `(实际掩码率, [MASK]占比, 随机替换占比, 保持原词占比)`。
掩码率 = 被计损失的位置数 ÷ 非特殊 token 数（对 n_trials 次取平均）。

In [ ]:
def mask_stats(ids, mask_prob, p_mask, p_random, n_trials=500):
    # TODO: 多次调用 mlm_mask，统计四个比例
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
base, _ = encode_pair(WORDS[:12])
rate, f_m, f_r, f_k = mask_stats(base, 0.15, 0.8, 0.1)
n_cand = (~np.isin(base, [CLS, SEP, PAD])).sum()
assert abs(rate - round(n_cand * 0.15) / n_cand) < 0.02, f'实际掩码率应≈15%，得到 {rate:.1%}'
assert abs(f_m - 0.8) < 0.05 and abs(f_r - 0.1) < 0.05
assert abs(f_m + f_r + f_k - 1.0) < 1e-9, '三者之和必须为 1'
# 换一套策略：100% 全 [MASK]（无补丁）
rate2, f_m2, f_r2, f_k2 = mask_stats(base, 0.15, 1.0, 0.0)
assert f_m2 > 0.99 and f_r2 < 0.01
# 更高掩码率（Wettig 2023 认为大模型适合 40%）
rate3, *_ = mask_stats(base, 0.40, 0.8, 0.1)
assert rate3 > 2 * rate, '掩码率参数应真实生效'
print(f'15% 策略: 掩码率 {rate:.1%}, mask/random/keep = {f_m:.0%}/{f_r:.0%}/{f_k:.0%}')
print(f'40% 策略: 掩码率 {rate3:.1%}')
print('✅ 练习 1 通过')

## ✏️ 练习 2：伪困惑度（pseudo-perplexity）

MLM 不定义合法的联合分布，但可以算一个**伪困惑度**作近似指标：
逐个位置掩掉一个 token，用其余全部上下文预测它，把所有位置的 log 概率平均后取指数。

实现 `pseudo_perplexity(model, ids)`：跳过特殊 token，返回 `exp(-mean log P(x_i | x_\i))`。

In [ ]:
def pseudo_perplexity(model, ids):
    # TODO: 对每个非特殊位置 i：把 ids[i] 换成 MASK，前向，取 softmax 后该位置对原词的概率
    #       返回 exp(-平均 log 概率)
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
test_ids, _ = encode_pair(['这家店', '不错', '值得', '再来'])
ppl = pseudo_perplexity(m_ok, test_ids)
assert 1.0 <= ppl <= V + 1e-6, f'伪困惑度应落在 [1, V]，得到 {ppl}'
# 未训练模型的伪困惑度应接近 V（随机猜）
ppl_rand = pseudo_perplexity(MiniBert(seed=99), test_ids)
print(f'训练过的模型: pseudo-PPL = {ppl:.2f}')
print(f'随机初始化  : pseudo-PPL = {ppl_rand:.2f}  (词表大小 V = {V})')
assert ppl_rand > V * 0.5, '随机模型的伪困惑度应接近词表大小'
print('\n⚠️  注意：伪困惑度**不能**与 GPT 的困惑度直接比较 ——')
print('    MLM 的一族条件分布互不相容，不存在以它们为条件边缘的联合分布。')
print('✅ 练习 2 通过')

## ✏️ 练习 3：堵死 NSP 的捷径

实现 `make_sop_data(n, seed)`：生成 **SOP** 数据。
给定一个由 8 个词组成的连续片段，正例是 `(前4词, 后4词, 1)`，负例是 `(后4词, 前4词, 0)`——
**同一对句子，只是顺序颠倒**。返回 `[(a, b, y), ...]`。

In [ ]:
def make_sop_data(n=400, seed=1):
    # TODO: 每条：从 TOPIC_A 或 TOPIC_B 采 8 个不重复的词作为「连续片段」
    #       50% 概率产出 (前4, 后4, 1)，否则 (后4, 前4, 0)
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
sop = make_sop_data(400, seed=2)
assert len(sop) == 400
ys = [y for _, _, y in sop]
assert 0.35 < np.mean(ys) < 0.65, '正负例应大致均衡'
# 关键性质：正负例的**词汇集合完全相同**，只是顺序不同 -> 主题捷径失效
acc = lexical_overlap_classifier(sop)
assert acc < 0.65, f'主题捷径在 SOP 上应失效（接近随机），得到 {acc:.1%}'
# 再验证：每条样本的 a∪b 都是 8 个不重复的词
for a, b, y in sop[:20]:
    assert len(set(a) | set(b)) == 8, '正负例应来自同一个 8 词片段'
print(f'SOP 上主题捷径的准确率 = {acc:.1%}（接近随机 50%）')
print('✅ 练习 3 通过：换一种负例采样，任务难度与所学信号完全不同')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def mask_stats(ids, mask_prob, p_mask, p_random, n_trials=500):
    n_cand = (~np.isin(ids, [CLS, SEP, PAD])).sum()
    n_masked = kinds = 0
    cnt = {'mask': 0, 'random': 0, 'keep': 0}
    for s in range(n_trials):
        c, lab = mlm_mask(ids, mask_prob=mask_prob, seed=s, p_mask=p_mask, p_random=p_random)
        sel = np.where(lab != -100)[0]
        n_masked += len(sel)
        for i in sel:
            if c[i] == MASK:      cnt['mask'] += 1
            elif c[i] != ids[i]:  cnt['random'] += 1
            else:                 cnt['keep'] += 1
    tot = sum(cnt.values())
    return (n_masked / (n_trials * n_cand),
            cnt['mask'] / tot, cnt['random'] / tot, cnt['keep'] / tot)

In [ ]:
# 练习 2 参考答案
def pseudo_perplexity(model, ids):
    logps = []
    for i in range(len(ids)):
        if ids[i] in (CLS, SEP, PAD):
            continue
        probe = ids.copy(); orig = probe[i]; probe[i] = MASK
        _, logits = model.forward(probe)
        p = softmax(logits[i])[orig]
        logps.append(math.log(p + 1e-12))
    return math.exp(-float(np.mean(logps)))

In [ ]:
# 练习 3 参考答案
def make_sop_data(n=400, seed=1):
    r = np.random.default_rng(seed)
    out = []
    for _ in range(n):
        topic = TOPIC_A if r.random() < 0.5 else TOPIC_B
        seg = list(r.choice(topic, size=8, replace=False))
        first, second = seg[:4], seg[4:]
        if r.random() < 0.5:
            out.append((first, second, 1))
        else:
            out.append((second, first, 0))
    return out

---
## 🧪 真实数据胶囊：BERT 的训练配方账

用 BERT 论文公开的配方数字，算三笔账：
① 两阶段序列长度省了多少注意力计算；② 掩码预算的实际有效部分；③ 相对 CLM 的信号量差距。

In [ ]:
# BERT 论文公开配方
STEPS_TOTAL = 1_000_000
FRAC_SHORT, LEN_SHORT, LEN_LONG = 0.9, 128, 512
BATCH_SEQ = 256

def attention_cost(seq_len, steps, batch):
    '''注意力的相对计算量 ∝ batch × steps × L²'''
    return batch * steps * seq_len ** 2

STEPS_SHORT = int(STEPS_TOTAL * FRAC_SHORT)
two_stage = (attention_cost(LEN_SHORT, STEPS_SHORT, BATCH_SEQ) +
             attention_cost(LEN_LONG,  STEPS_TOTAL - STEPS_SHORT, BATCH_SEQ))
all_long  = attention_cost(LEN_LONG, STEPS_TOTAL, BATCH_SEQ)
print(f'① 两阶段序列长度 vs 全程 512:')
print(f'   两阶段 {two_stage:.3e}  |  全程512 {all_long:.3e}  ->  省 {(1-two_stage/all_long):.0%}')
assert two_stage < 0.3 * all_long, '两阶段应省下约 80% 以上的注意力计算'

print(f'\n② 掩码预算的有效部分:')
budget = 0.15
effective = budget * 0.9        # 扣掉 10% 保持原词（近乎无信息）
print(f'   名义掩码率 {budget:.0%} -> 有效信号位置 ≈ {effective:.1%}')

print(f'\n③ 相对 CLM 的信号量:')
tok_per_step = BATCH_SEQ * LEN_SHORT
print(f'   MLM 每步有效预测数 ≈ {tok_per_step * effective:,.0f}')
print(f'   CLM 每步预测数     ≈ {tok_per_step:,.0f}   ({1/effective:.1f}× )')
assert 1 / effective > 6, 'MLM 的信号密度劣势应在 6-7 倍量级'
print('\n✅ 这就是「BERT 需要百万步训练」的定量解释，也是模块 02 里 ELECTRA 的动机。')

**🧪 胶囊练习**：实现 `optimal_two_stage(total_steps, frac_short, len_short, len_long)`：
返回 `(注意力相对计算量, 相对全程长序列的节省比例)`。
再验证：`frac_short` 越大，节省越多（但位置嵌入的长位置见得越少——这是权衡）。

In [ ]:
def optimal_two_stage(total_steps, frac_short, len_short, len_long, batch=256):
    # TODO: cost = batch*steps_short*len_short^2 + batch*steps_long*len_long^2
    #       返回 (cost, 1 - cost/全程长序列的 cost)
    raise NotImplementedError

In [ ]:
# 自测
c, saving = optimal_two_stage(1_000_000, 0.9, 128, 512)
assert abs(saving - (1 - two_stage / all_long)) < 1e-6
savings = [optimal_two_stage(1_000_000, f, 128, 512)[1] for f in [0.0, 0.5, 0.9, 0.99]]
assert savings == sorted(savings), 'frac_short 越大，节省越多'
assert savings[0] == 0.0, 'frac_short=0 时就是全程长序列，无节省'
print('frac_short:', [0.0, 0.5, 0.9, 0.99])
print('节省比例  :', [f'{s:.0%}' for s in savings])
print('✅ 胶囊练习通过：但 frac_short 太大会让长位置嵌入训练不足 —— 90% 是经验平衡点')

In [ ]:
# 📖 胶囊参考答案
def optimal_two_stage(total_steps, frac_short, len_short, len_long, batch=256):
    s_short = int(total_steps * frac_short)
    s_long = total_steps - s_short
    cost = batch * s_short * len_short ** 2 + batch * s_long * len_long ** 2
    full = batch * total_steps * len_long ** 2
    return cost, 1 - cost / full

---
## 🔧 旁注：真实库里这些对应什么

- **三种嵌入相加** → `transformers.BertEmbeddings`（`word_embeddings + token_type_embeddings + position_embeddings`）。
- **MLM 掩码 80/10/10** → `DataCollatorForLanguageModeling(tokenizer, mlm=True, mlm_probability=0.15)`，源码里就是这三档。
- **labels=-100** → PyTorch `CrossEntropyLoss(ignore_index=-100)` 的约定；HF 全生态沿用。
- **[CLS] 池化** → `BertPooler`（`tanh(W·h_cls)`）；`BertForSequenceClassification` 用它。做句嵌入时通常改用 mean pooling（模块 05）。
- **512 长度上限** → `config.max_position_embeddings`；超了会报索引错误，不是效果变差。
- **两阶段序列长度** → 预训练脚本里的 `--max_seq_length` 分阶段设置。

想真正跑起来（加载权重、`Trainer`、`DataCollator`），见 **C50**；本课只负责让你**知道每一行在做什么**。

### 小结
- 双向表示与自回归目标**不可兼得**：MLM 通过「把答案从输入里挖掉」拿到双向上下文。
- **80/10/10 是补丁**，补的是「`[MASK]` 在下游从不出现」这个预训练-微调不匹配；ELECTRA 从根上绕开（模块 02）。
- **损失只在被掩位置计算**；对所有位置计损失会让 loss 更低但学到抄写捷径——**低 loss ≠ 好表示**。
- **NSP 被推翻**，因为它能被「主题匹配」捷径攻破；SOP 用同主题负例堵死了这条路。方法论：设计自监督任务先找捷径。
- BERT 不能生成有三层原因，最深的一层是 **MLM 不定义合法的联合分布**（伪困惑度不可与 CLM 困惑度比较）。
- **信号密度 0.15×** 是 MLM 的核心代价，但要乘上每次信号的信息量才是公平比较。

下一站：**模块 02 · 预训练目标的改良** —— 同一个架构，把配方与目标换一遍，效率能差几倍。